# Objetivo

Este laboratorio es diagnóstico. Su propósito es verificar que puedes abrir el entorno y conectar cuatro niveles que usaremos durante todo el curso: población, muestra aleatoria, estimador y valor calculado. Después ejecutarás un primer flujo de clasificación y escribirás una interpretación técnica breve.

Trabajaremos con el dataset `load_wine` de `scikit-learn`. El objetivo será clasificar vinos en tres clases a partir de mediciones químicas.

No necesitas entender aún la matemática de la regresión logística. La usaremos como modelo de referencia; el objeto matemático central de esta actividad es la relación entre riesgo esperado y riesgo empírico.

# Entrega

Entrega preferiblemente el archivo `semana01_lab.ipynb` ejecutado. Si trabajas desde esta versión Quarto, copia el código y tus respuestas a un notebook o a un documento con resultados visibles.

Nombre recomendado:

```text
apellido_nombre_semana01_lab.ipynb
```

La entrega debe incluir:

- código ejecutado;
- salidas principales;
- respuestas de interpretación;
- conclusión técnica;
- dos ideas iniciales de proyecto;
- declaración breve de uso de IA, si aplica.

# Parte 0: preparación

Ejecuta las siguientes importaciones. Si esta celda falla, revisa `recursos/configuracion_entorno.qmd`.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Parte 1: población, muestra y riesgo

Considera una población en la que

$$
Y\sim\operatorname{Bernoulli}(0.7).
$$

Una regla $f_1$ predice siempre 1 y usa pérdida cero-uno. Su riesgo esperado es

$$
R_P(f_1)=\mathbb P(Y\neq 1)=0.3.
$$

Genera una muestra de tamaño 20 y calcula el riesgo empírico.

In [ ]:
rng = np.random.default_rng(2105)
p_clase_1 = 0.7
m = 20

muestra = rng.binomial(1, p_clase_1, size=m)
riesgo_esperado = 1 - p_clase_1
riesgo_empirico = np.mean(muestra != 1)

print("Muestra:", muestra)
print(f"Riesgo esperado: {riesgo_esperado:.3f}")
print(f"Riesgo empírico: {riesgo_empirico:.3f}")

Ahora repite el muestreo 2000 veces.

In [ ]:
riesgos_repetidos = np.array([
    np.mean(rng.binomial(1, p_clase_1, size=m) != 1)
    for _ in range(2_000)
])

media_simulada = riesgos_repetidos.mean()
sd_simulada = riesgos_repetidos.std(ddof=1)
se_teorico = np.sqrt(riesgo_esperado * (1 - riesgo_esperado) / m)

print(f"Media de riesgos empíricos: {media_simulada:.3f}")
print(f"SD simulada: {sd_simulada:.3f}")
print(f"Error estándar teórico: {se_teorico:.3f}")

Auto-chequeo esperado:

- `riesgo_esperado` debe ser 0.3;
- la media simulada debe quedar cerca de 0.3;
- la desviación simulada debe quedar cerca del error estándar teórico;
- repetir con la misma semilla debe producir los mismos números.

## Pregunta probabilística

Responde con cuatro frases separadas:

1. ¿Cuál es el objeto poblacional que queremos conocer?
2. ¿Cuál es la muestra aleatoria y cuál es su realización observada?
3. ¿Cuál es el estimador y cuál es la estimación obtenida en la primera muestra?
4. ¿Por qué `riesgo_empirico` no tiene que ser exactamente 0.3?

**Respuesta:**

```text
Escribe aquí.
```

# Parte 2: cargar e inspeccionar datos

In [ ]:
data = load_wine(as_frame=True)

X = data.data
y = data.target

X.head()

Responde:

1. Qué representa una fila?
2. Cuántas variables de entrada tiene el dataset?
3. Cuántas clases hay?

In [ ]:
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", data.target_names)
print("Conteo por clase:")
print(y.value_counts().sort_index())

Auto-chequeo esperado:

- `X` debe tener 178 filas y 13 columnas.
- `y` debe tener 178 valores.
- Los conteos por clase deben ser 59, 71 y 48.

## Pregunta de interpretación 1

Escribe dos frases:

- una sobre la estructura de los datos;
- otra sobre si las clases parecen balanceadas.

**Respuesta:**

```text
Escribe aquí.
```

# Parte 3: partición de datos

Vamos a separar entrenamiento y prueba.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2105,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Clases en entrenamiento:")
print(y_train.value_counts(normalize=True).sort_index())
print("Clases en prueba:")
print(y_test.value_counts(normalize=True).sort_index())

Auto-chequeo esperado:

- El conjunto de entrenamiento debe tener 133 observaciones.
- El conjunto de prueba debe tener 45 observaciones.
- Las proporciones de clase deben ser parecidas en entrenamiento y prueba.

## Pregunta de interpretación 2

Por qué usamos `stratify=y`? Responde en una o dos frases.

**Respuesta:**

```text
Escribe aquí.
```

# Parte 4: baseline

Antes de entrenar un modelo, necesitamos una referencia simple.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)

baseline_acc

Auto-chequeo esperado:

- El baseline predice siempre la clase más frecuente en entrenamiento.
- Con esta partición, la accuracy del baseline debería estar alrededor de 0.40.

## Pregunta de interpretación 3

Qué significa este baseline? Qué tendría que lograr un modelo para ser mejor que esta referencia?

**Respuesta:**

```text
Escribe aquí.
```

# Parte 5: modelo de referencia

Entrenaremos una regresión logística con escalamiento de variables.

In [ ]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

model.fit(X_train, y_train)
model_pred = model.predict(X_test)
model_acc = accuracy_score(y_test, model_pred)

model_acc

Compara:

In [ ]:
print(f"Baseline accuracy: {baseline_acc:.3f}")
print(f"Model accuracy:    {model_acc:.3f}")
print(f"Improvement:       {model_acc - baseline_acc:.3f}")

Auto-chequeo:

- El modelo debería superar claramente al baseline.
- Si no lo supera, revisa si ejecutaste las celdas en orden y si el pipeline incluye `StandardScaler`.

# Parte 6: errores

La accuracy resume desempeño en un número, pero no muestra que errores comete el modelo.

In [ ]:
cm = confusion_matrix(y_test, model_pred)
cm

In [ ]:
print(classification_report(y_test, model_pred, target_names=data.target_names))

## Pregunta de interpretación 4

Observa la matriz de confusion y el reporte. Escribe:

1. Qué clase parece más fácil o más difícil de clasificar?
2. Qué limitacion tiene esta conclusión?

**Respuesta:**

```text
Escribe aquí.
```

# Parte 7: accuracy como estimación y conclusión técnica

La accuracy observada es un promedio de indicadores de acierto. Calcula su error estándar *plug-in*:

In [ ]:
model_acc_se = np.sqrt(model_acc * (1 - model_acc) / len(y_test))
print(f"Accuracy observada: {model_acc:.3f}")
print(f"Error estándar plug-in: {model_acc_se:.3f}")

Esta cuenta supone observaciones de test independientes y una misma probabilidad de acierto. No incorpora cambio de población, búsqueda de modelos ni dependencia entre observaciones.

Completa el siguiente párrafo. Debe mencionar baseline, métrica, incertidumbre, datos y limitación.

> En este experimento inicial, el modelo de referencia supera / no supera el baseline porque ____. La accuracy observada es ____ y su error estándar *plug-in* es ____. Esta cantidad intenta estimar ____. Sin embargo, la conclusión está limitada por ____.

**Respuesta final:**

```text
Escribe aquí.
```

# Parte 8: dos ideas iniciales de proyecto

Propone dos áreas o datasets que podrían interesarte para el proyecto final.

Para cada idea responde:

1. ¿Cuál sería la población de interés?
2. ¿Cuál sería la unidad de observación y cómo se obtendría la muestra?
3. ¿Cuál sería la pregunta aplicada?
4. ¿Habría variable objetivo? Si sí, ¿cuál?
5. ¿Qué cantidad poblacional o riesgo intentaría estimar?
6. ¿Qué limitación o fuente de incertidumbre anticipas?

## Idea 1

```text
Escribe aquí.
```

## Idea 2

```text
Escribe aquí.
```

Este laboratorio es diagnóstico. La prioridad es que muestres el flujo de trabajo y que identifiques dudas temprano.

# Checklist antes de entregar

Marca mentalmente cada punto:

- Pude importar los paquetes o documente el error.
- Distinguí población, muestra, estimador y estimación.
- Comparé riesgo esperado y riesgo empírico mediante simulación.
- Identifique filas, columnas y clases.
- Explique por qué se usa `stratify=y`.
- Calcule baseline y modelo.
- Compare modelo contra baseline.
- Escribí una conclusión con métrica, incertidumbre, datos, baseline y limitación.
- Propuse dos ideas iniciales de proyecto.
- Declare uso sustancial de IA, si aplica.